**ML1-Notebook5**

***Açelya Yıldırım (Group 17)***

***26.10.2025***

# Cross-validation

With the code developed so far, it is possible to train an ANN and provide an estimate of the results it would offer in its real execution (with unseen patterns, represented by a test set). However, in this last aspect there are two factors to consider, as a consequence of the non-deterministic nature of the process we are following:

- The partitioning of the set of patterns into training/test is random (hold out), and is therefore overly dependent on good or bad luck in choosing training and test patterns.
- ANN training is not deterministic, as the initialisation of the weights is random. As before, it is too dependent on good or bad luck to start the training at a good or bad starting point.

For these two reasons, the test result of a single training is not significant when assessing the goodness of fit of the model in the presence of unseen patterns. To solve this problem, the experiment is repeated several times and the results are averaged. This can be implemented in a simple way by means of a loop; however, it is necessary to do this in an orderly way as there are two different sources of randomness.

Firstly, to minimise the randomness due to the partitioning of the data set, it is necessary to have a method that ensures that each data is used for training at least once, and for testing at least once. The most commonly used method is cross-validation. In this method, the data set is split into k disjoint subsets and k experiments are performed. In the k-th experiment, the k subset is separated for testing, and the remaining k-1 substes are used for training, performing a k-fold cross-validation. A common value is k=10, which gives a 10-fold cross-validation. Finally, the test value corresponding to the appropriate metric will be the average value of the values of the k experiments.

A widely used variant of cross-validation is stratified cross-validation. In this case, each subset is created in such a way as to keep the proportion of patterns of each class the same (or similar) as in the original dataset. This is particularly used when the data set is imbalanced.

It is usual to save not only the mean, but also the k values, in order to subsequently perform a paired hypothesis test with another model. To do this, it is necessary that both models have been trained using the same training and test sets.

This way of evaluating the model is often considered to be slightly pessimistic, i.e. the results obtained in tests are slightly worse than those that would be obtained from real training with all available data. In a hold out experiment, as mentioned above, several data are separated for testing. This means that the model is trained with less data than is available, and that by chance the data separated for testing can be of great importance (especially if there is little data). For this reason, when training with less data and possibly no "important" data, hold out is considered a pessimistic assessment. In the same way, cross-validation also separates data for testing, so it does not train on all available data, and is therefore also pessimistic. However, it is guaranteed that all data are used at least once in training and once in testing, thus trying to minimise the impact of chance in separating data, so it is considered only a slightly pessimistic evaluation.

Doing this is as simple as splitting the data set and performing a loop with k iterations in which at the k-th iteration a model is trained and evaluated with the corresponding sets. However, if the model is not deterministic, the result obtained at the k-th iteration will not be meaningful, since it is again dependent on chance. In this case, what needs to be done is a second nested loop within iteration k in which the model is repeatedly trained, and finally an average of the results is made to finally output the result of iteration k. The number of trainings must be high for the average results to be really significant, at least 50 trainings.

### Question 5.1

> ❓ If this second loop is performed with a deterministic model, what will be the standard deviation of the test results obtained? Is there a difference between performing this second loop and averaging the results, or doing a single training?

In a deterministic model, given the same input data and initial conditions, 
the model will always produce identical outputs. This means:

Standard Deviation will be 0: If we train the same deterministic model multiple times on the same training data, we will get the same results every time. Therefore, the standard deviation of test results across these repeated trainings will be zero.

No difference between single and multiple trainings: For deterministic models, performing the inner loop (multiple executions per fold) and averaging the results is equivalent to doing a single training. Both approaches yield the same outcome since there is no randomness in the training process.

Practical implication: The inner loop (numExecutions) is only necessary for non-deterministic models like ANNs, where random weight initialization causes variation between training runs. For deterministic models (e.g., Decision Trees, SVM, kNN), this inner loop is redundant and computationally wasteful.

In summary, the standard deviation of deterministic models is 0. And we don't need to perform multiple executions per fold.

In this way, it is possible to evaluate a model together with its hyperparameters in solving a problem. A very common situation is to compare several models (or the same model with different hyperparameters), for which this scheme has to be applied with an important caveat: the sets used in the cross-validation must be the same for each model. Since the distribution of patterns in different sets is random, having the same subsets in different runs is achieved by setting the random seed at the beginning of the program to be executed. Setting the random seed not only allows the same subsets to be generated, but is also important in order to be able to repeat the results in different runs.

It is also important to bear in mind that this methodology allows estimating the real performance of a model (although slightly pessimistic). The final model that would be used in production would be the result of training it with all the available patterns, since, as seen in the theory class, and very generally speaking, the more patterns you train with, the better the model will be.

Another commonly requested task is to obtain a **confusion matrix on the test set** as a result of performing **cross-validation**.

In general, this is **not directly possible**, because multiple neural networks are trained for each fold.  
If only a single model were trained per fold (as will be the case in the next exercise), it would be possible to compute a confusion matrix for each test set and then sum all of them to obtain a **global test confusion matrix**.

However, since multiple models are trained in each fold here, we use an alternative approach:

- For each fold, a **confusion matrix is computed for every execution**.
- These matrices are then **averaged within each fold**.
- Finally, the averaged matrices from each fold are **summed** to produce a **global confusion matrix**.

---

### ⚠️ Important considerations

- This global confusion matrix **does not represent the result of any single model**.  
  Therefore, calling it a "confusion matrix" can be misleading — but it is still useful to analyze how instances are distributed in the test sets.

- For the same reason, the **metrics derived from this matrix** will generally **not match** the metric values returned by the function (which are averages of the metric scores for each fold).  
  These matrices are meant to provide **a visual and aggregated summary**, while the true evaluation metrics are the ones computed per fold.

- As a result of averaging the confusion matrices, the **final confusion matrix will contain real numbers**, not integers.  
  While this is technically not correct, it is acceptable to include it in a report — as long as the nature of the matrix is clearly explained.

In this assignment, you are asked to:

1. Develop a function called `crossvalidation` that receives a value `N` (equal to the number of patterns), and a value `k` (number of subsets into which the dataset is to be split), and returns a vector of length N, where each element indicates in which subset that pattern should be included.

    To perform this function, one possibility is to perform the following steps:
    
    - Create a vector with k sorted elements, from 1 to k.
    - Create a new vector with repetitions of the previous vector until its length is greater than or equal to N. The functions `repeat` and `ceil` can be used for this purpose.
    - Take the first N values of this vector.
    - Shuffle this vector (using the function `shuffle!` and return it. To use this function, the module `Random` should be loaded.
    
    This function should return a vector of **integer values**, with a length equal to **N**. No loop should be used in this function.

In [3]:
using Random

function crossvalidation(N::Int64, k::Int64)
    # Step 1: Create base vector [1, 2, 3, ..., k]
    baseVector = 1:k
    
    # Step 2: Repeat it enough times to cover N patterns
    # We need ceil(N/k) repetitions to ensure length >= N
    repeatedVector = repeat(baseVector, ceil(Int, N/k))
    
    # Step 3: Take only the first N elements
    indices = repeatedVector[1:N]
    
    # Step 4: Shuffle randomly and return
    return shuffle!(indices)
end

crossvalidation (generic function with 1 method)

2. Develop a new function called `crossvalidation` that receives:

- A vector `targets::AbstractArray{Bool,1}` containing the desired outputs (ground truth labels),
- An integer `k`, which is the number of folds for the partitioning.

The function must return a vector of **integer values** of length **N** (where `N` is the number of elements in `targets`), indicating to which subset (fold) each instance belongs.

The partitioning must be **stratified**, meaning that **the distribution of positive and negative instances is preserved across folds**. 
Follow these steps:

    1. Create a vector of indices with as many elements as there are rows in `targets`.
    2. Call the previously developed `crossvalidation` function, passing the **number of positive instances** and the value of `k`.
    3. Assign the result of the previous step to the positions of the index vector that correspond to **positive instances**.
  ### Question 5.2
  > ❓ Could you combine steps 2 and 3 into a single line? (This is not required, but worth considering.)

Yes, we can combine steps 2 and 3 into a single line using Julia's indexing syntax:

indexes[targets] = crossvalidation(sum(targets), k)

This single line:
1. Counts positive examples using sum(targets)
2. Creates CV indices for them using the previous cross-validation function
3. Assigns these indices to the positions where the targets are true

This approach is more concise, follows Julia's idiomatic style, and leverages Boolean indexing for elegant array manipulation.

    4. Repeat a similar operation for **negative instances**.
    5. Return the index vector.

Loops are **not allowed** in the implementation of this function. The function must return a **vector of integers** of length **N**.

In [5]:
function crossvalidation(targets::AbstractArray{Bool,1}, k::Int64)
    # Create index vector with same length as targets
    indices = zeros(Int64, length(targets))
    
    # For positive class (true values) - combined in one line
    indices[targets] = crossvalidation(sum(targets), k)
    
    # For negative class (false values) - combined in one line
    indices[.!targets] = crossvalidation(sum(.!targets), k)
    
    return indices
end

crossvalidation (generic function with 2 methods)

3. Develop a new function called `crossvalidation` that receives:

- A matrix `targets::AbstractArray{Bool,2}`, where each row corresponds to one instance and each column to one class (multilabel format),
- An integer `k`, indicating the number of folds (subsets) to divide the dataset into.

The function must return a vector of integers of length `N` (where `N` is the number of rows in `targets`).  
Each element in the vector indicates the subset (fold) that the corresponding instance should be assigned to.

The partitioning must be **stratified**, meaning that the distribution of **each class across the folds is preserved**.
Suggested Steps:

    1. Create an index vector with as many elements as rows in `targets`.

    2. Use a loop that **iterates over the classes** (i.e., the columns of `targets`).  
     For each class:

       i. Count the number of instances that belong to that class using `sum(targets[:, i])`.  
       ii. Call the previously defined `crossvalidation` function with the number of instances and `k`.  
       iii. Assign the result to the corresponding positions in the index vector where `targets[:, i]` is `true`.

   ### Question 5.3
   > ❓ Could you combine these three operations into a single line inside the loop? (Optional exercise)

Yes, we can combine counting instances, generating CV indices, and assignment into a single line:

indices[targets[:, class]] = crossvalidation(sum(targets[:, class]), k)

This elegant one-liner:
1. Counts instances in the class using sum(targets[:, class])
2. Generates CV indices for that count by calling the base cross-validation function
3. Assigns the result to positions where targets[:, class] is true

This approach follows Julia's functional programming paradigm and makes the code more concise while maintaining readability.

    3. Return the index vector.

  #### Important considerations

   - This function is allowed to use **a single loop** that iterates over the classes.
   - The result must be a vector of **integer values** with length equal to the number of instances `N`.

  #### Class support per fold

  It is essential to ensure that **each class has at least `k` instances**.

  > A typical value is `k = 10`.

### Question 5.4
> ❓ What would happen if a class has fewer than `k` instances?

If a class has fewer than k instances, serious problems occur. They could be like;

Impossible stratification: We cannot create k folds where each fold contains at least one instance of that class. Some folds will have zero instances of the underrepresented class.

Empty test sets for some classes: In certain folds, the test set will contain no instances of the underrepresented class, making it impossible to evaluate the model's performance on that class for those folds.

Training set issues: Some training sets might completely lack examples of the minority class, preventing the model from learning to recognize it at all.

Biased results: The cross-validation results become unreliable as not all classes are fairly represented across folds.

### Question 5.5
> ❓ How would this affect the calculation of metrics such as sensitivity or specificity?

When a class has fewer than k instances, metric calculations become problematic:

1. Sensitivity (Recall) = TP/(TP+FN): For folds where the test set contains no positive instances of a class, the denominator (TP+FN) equals zero, resulting in an undefined mathematical operation. This produces a NaN value in computational systems.

2. Specificity = TN/(TN+FP): Similarly, if a fold's test set has no negative instances, the denominator becomes zero, yielding NaN.

3. PPV (Precision) = TP/(TP+FP): When the model fails to predict a minority class due to insufficient training examples, both TP and FP may be zero, causing division by zero and resulting in NaN.

4. F1-Score: Being the harmonic mean of precision and recall, F1-score inherits any undefined values from its components, propagating NaN through the calculation.

5. Averaging across folds: When computing mean metrics, NaN values propagate through arithmetic operations, contaminating the final aggregated results and rendering the cross-validation evaluation invalid.

Impact: The reported performance metrics fail to accurately represent the model's true predictive capability, particularly for underrepresented classes, leading to misleading conclusions about model quality.

Conclusion: Ensuring adequate class representation (i.e., at least k instances per class) is crucial for mathematically valid metric computation and reliable cross-validation assessment.

If, for any reason, it is not possible to guarantee at least `k` samples per class, you may consider **reducing the value of `k`**.  
In this case, consult your instructor to evaluate the implications and whether the trained models would still yield meaningful results.

In [7]:
function crossvalidation(targets::AbstractArray{Bool,2}, k::Int64)
    # Get number of patterns (rows)
    N = size(targets, 1)
    
    # Initialize index vector
    indices = zeros(Int64, N)
    
    # Iterate over each class (column)
    for class in 1:size(targets, 2)
        # Single line: count instances, generate CV indices, and assign
        indices[targets[:, class]] = crossvalidation(sum(targets[:, class]), k)
    end
    
    return indices
end

crossvalidation (generic function with 3 methods)

4. Implement a final version of the `crossvalidation` function, where:

- The first argument is `targets::AbstractArray{<:Any,1}`, a vector containing **heterogeneous elements** (e.g., strings, symbols, integers, etc.),
- The second argument is the usual integer `k` (number of folds),
- The output is a vector of **integer values**, one per instance, indicating the fold assignment.

To implement this function, follow these steps:

   1. Call the `oneHotEncoding` function, passing `targets` as the input.  
      This will convert the labels into a binary matrix (multilabel format).
   2. Call the **previous version** of the `crossvalidation` function, passing the encoded matrix and the value `k`.

   > **No loops are allowed** in this function.
   > The returned vector must contain integers and have the same length as the input.

In [9]:
function crossvalidation(targets::AbstractArray{<:Any,1}, k::Int64)
    # Convert heterogeneous labels to one-hot encoded Boolean matrix
    oneHotTargets = oneHotEncoding(targets)
    
    # Call the multi-class cross-validation function (Bool,2)
    return crossvalidation(oneHotTargets, k)
end

crossvalidation (generic function with 4 methods)

5. Finally, implement a function called `ANNCrossValidation` that trains an artificial neural network (ANN) multiple times following a **cross-validation strategy**, as described in this exercise.

 This function will receive both the usual inputs for training a neural network (as seen in previous exercises) and additional arguments for performing cross-validation with multiple executions per fold.

   #### **Function parameters**

   - `topology::AbstractArray{<:Int,1}` — Structure of the neural network.
   - `dataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{<:Any,1}}` — A tuple with:
     - A matrix of inputs (instances in rows),
     - A vector of target labels (categorical). These must be converted into a Boolean matrix via `oneHotEncoding` inside this function.
   - `crossValidationIndices::Array{Int64,1}` — Vector of cross-validation fold assignments for each instance, obtained via the `crossvalidation` function.

   **Optional parameters** (with default values, as specified in the function signature):

   - `numExecutions`: Number of training repetitions per fold.
   - `transferFunctions`: Activation functions for each layer in the ANN.
     ### Question 5.6
     > ❓ Does it make sense to use a linear activation function in hidden layers?
    

No, using linear activation functions in hidden layers does not make sense:

1. Loss of non-linearity: Linear activations cannot model non-linear relationships in data. The primary advantage of neural networks is their ability to learn complex, non-linear patterns through non-linear activation functions.

2. Collapse to a single layer: Multiple linear transformations compose into a single linear transformation:
   - If f(x) = x (linear), then f(W₂ · f(W₁ · x)) = W₂ · W₁ · x = W_combined · x
   - This reduces a deep network to effectively a single-layer linear model, eliminating the benefits of depth.

3. No representation power: A multi-layer network with linear activations has the same representational capacity as logistic/linear regression, defeating the purpose of using neural networks.

4. When linear is appropriate:
   - Output layer for regression: When predicting continuous values
   - Never in hidden layers for learning complex patterns

5. Proper choice: Hidden layers should use non-linear functions like sigmoid (σ), tanh, ReLU, or leaky ReLU to enable the network to approximate any continuous function (Universal Approximation Theorem).

Conclusion: We should always use non-linear activation functions in hidden layers to maintain the network's ability to learn complex decision boundaries and patterns.

   - `maxEpochs`: Maximum number of training iterations (epochs).
   - `minLoss`: Minimum loss to stop training early.
   - `learningRate`: Learning rate.
   - `validationRatio`: Fraction of data used for validation in early stopping (can be 0).
   - `maxEpochsVal`: Maximum number of epochs without validation improvement before stopping early.


 **Step-by-step implementation**

   1. **Extract class labels**:
      ```julia
      classes = unique(targets)
      ```
   2. **One-hot encode** the categorical target labels using the `oneHotEncoding` function and the computed `classes`.
      ### Question 5.7
      > ❓ What could go wrong if the `classes` vector is not passed explicitly to `oneHotEncoding`?

If the classes vector is not passed explicitly to oneHotEncoding, several critical problems occur during cross-validation:

1. Inconsistent encoding across folds: If oneHotEncoding infers classes independently for each fold, different folds may have different class orderings. 

For example, Fold 1 might encode as [ClassA, ClassB, ClassC] while Fold 2 encodes as [ClassB, ClassA, ClassC], breaking cross-fold comparability.

2. Dimension mismatch: Different folds might have different numbers of classes if some classes are missing from certain training sets:
   - Fold 1 training set: 3 classes → 3 output columns
   - Fold 2 training set: only 2 classes present → 2 output columns
   - This causes matrix dimension errors when trying to train or compare models

3. Missing class representation: If a fold's training set lacks a particular class entirely (common with small datasets or rare classes), oneHotEncoding won't create a column for it, making the network unable to predict that class.

4. Confusion matrix incompatibility: Cannot accumulate confusion matrices across folds if they have different dimensions or class orderings. The global confusion matrix becomes impossible to compute.

5. Metric calculation errors: Sensitivity, specificity, and other per-class metrics become meaningless when class indices differ across folds.

Solution: We should always extract classes from the full dataset before cross-validation:
```julia
classes = unique(targets)
oneHotTargets = oneHotEncoding(targets, classes)
```
This ensures every fold uses identical encoding schemes with the same number of columns and class ordering, making results comparable and metrics calculable.

Example: Dataset has ["A", "B", "C"], but Fold 2 training only sees ["A", "B"]
- Without explicit classes: Creates only 2 columns (wrong)
- With explicit classes: Creates 3 columns, with column 3 all zeros (correct)

   3. **Determine the number of folds** using:
      ```julia
      numFolds = maximum(crossValidationIndices)
      ```
   4. Create one vector per metric (`accuracy`, `error rate`, `sensitivity`, `specificity`, `PPV`, `NPV`, `F1`) to store fold results.

   5. Initialize a **confusion matrix accumulator** (matrix of real numbers with all entries set to 0) for the **global test confusion matrix**.

   **For each fold**:

   - Extract `train` and `test` subsets for **inputs** and **outputs**, based on the cross-validation indices and current fold number.

   - Since ANNs are **non-deterministic**, results from a single training per fold may not be representative.  
     For this reason, train the ANN **multiple times per fold** (as specified in `numExecutions`).

   **Inside each fold**:

   1. Initialize vectors to store the metric results for each execution.  
   2. Create a 3D array of size `(numClasses, numClasses, numExecutions)` to store test confusion matrices from each execution.

   3. For each execution:

      - If `validationRatio > 0`, split the training set into training and validation sets using `holdOut`.  
        > ⚠️ You must adjust the ratio properly, since you’re applying it **on the training subset**, not the full dataset.
        ### Question 5.8  
        > ❓ How can you adapt the validation ratio accordingly?

The validationRatio parameter represents the proportion of data to use for validation. In cross-validation, we must decide whether to interpret this ratio relative to the original dataset or relative to the current training fold.

The Situation:
After splitting data into k folds, the training set contains (k-1)/k of the original data. When we apply validationRatio, we need to clarify the reference point.

Two Interpretation Approaches:

1. Direct application (simpler):
   We can apply validationRatio directly to the training subset:
```julia
   holdOut(size(trainingInputs, 1), validationRatio)
```
   - If validationRatio = 0.1, we take 10% of the training fold for validation
   - This is 10% × 90% = 9% of the original dataset (for k=10)

2. Proportional adjustment (more precise):
   Adjust the ratio to maintain the same proportion relative to the original dataset:
```julia
   adjustedRatio = validationRatio * k / (k-1)
   holdOut(size(trainingInputs, 1), adjustedRatio)
```
   - If validationRatio = 0.1 and k=10, adjusted = 0.1 × 10/9 ≈ 0.111
   - This gives exactly 10% of the original dataset for validation

Which to use?
Both approaches are valid. The direct application is simpler and commonly used in practice, as the difference is minimal, and validation serves primarily for early stopping rather than precise proportioning. However, if the exact dataset proportions are critical, the adjusted approach ensures consistency.

For implementation: Unless specified otherwise, the direct application is standard practice and sufficient for this exercise.

      - Train the network using `trainClassANN`.

      - Evaluate it on the test set using `confusionMatrix`.

      - Store the returned metrics and confusion matrix.

   4. After all executions for this fold:

      - Compute the **average** of each metric vector and store it in the global metric vectors.
      - Compute the **mean confusion matrix** using:
        ```julia
        mean(confusionMatrices, dims=3)
        ```
        > Note: This returns a 3D array with one slice; you must extract the 2D matrix from it.

      - Add the resulting matrix to the global confusion matrix.

#### Return values

Once all folds are completed:

- For each metric, compute the **mean and standard deviation** across the folds using `mean(...)` and `std(...)`.
- Return a tuple with 8 elements:
  1. Accuracy (mean, std)
  2. Error rate (mean, std)
  3. Sensitivity (mean, std)
  4. Specificity (mean, std)
  5. PPV (mean, std)
  6. NPV (mean, std)
  7. F1-score (mean, std)
  8. Global test confusion matrix

This function will be called in the next exercise by a general-purpose validation function.  
Unlike ANNs, **other ML models (e.g., SVM, kNN)** are **deterministic** — so they do not need the inner execution loop.  
For them, training once per fold is sufficient to produce stable results.

In [16]:
function ANNCrossValidation(topology::AbstractArray{<:Int,1},
        dataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{<:Any,1}},
        crossValidationIndices::Array{Int64,1};
        numExecutions::Int=50,
        transferFunctions::AbstractArray{<:Function,1}=fill(σ, length(topology)),
        maxEpochs::Int=1000, minLoss::Real=0.0, learningRate::Real=0.01,
        validationRatio::Real=0, maxEpochsVal::Int=20)
    
    # 1. Extract inputs and targets from dataset
    inputs, targets = dataset
    
    # 2. Get unique classes and encode targets
    classes = unique(targets)
    oneHotTargets = oneHotEncoding(targets, classes)
    
    # 3. Determine number of folds and classes
    numFolds = maximum(crossValidationIndices)
    numClasses = length(classes)
    
    # 4. Initialize metric vectors for each fold
    testAccuracies = zeros(Float64, numFolds)
    testErrorRates = zeros(Float64, numFolds)
    testSensitivities = zeros(Float64, numFolds)
    testSpecificities = zeros(Float64, numFolds)
    testPPVs = zeros(Float64, numFolds)
    testNPVs = zeros(Float64, numFolds)
    testF1Scores = zeros(Float64, numFolds)
    
    # 5. Initialize global confusion matrix accumulator
    globalConfusionMatrix = zeros(Float64, numClasses, numClasses)
    
    # 6. OUTER LOOP: Iterate over each fold
    for numFold in 1:numFolds
        
        # 7. Split data into training and test sets for this fold
        # Use Boolean masks
        trainingInputs = inputs[crossValidationIndices .!= numFold, :]
        testInputs = inputs[crossValidationIndices .== numFold, :]
        trainingTargets = oneHotTargets[crossValidationIndices .!= numFold, :]
        testTargets = oneHotTargets[crossValidationIndices .== numFold, :]
        
        # 8. Initialize metric vectors for executions within this fold
        executionAccuracies = zeros(Float64, numExecutions)
        executionErrorRates = zeros(Float64, numExecutions)
        executionSensitivities = zeros(Float64, numExecutions)
        executionSpecificities = zeros(Float64, numExecutions)
        executionPPVs = zeros(Float64, numExecutions)
        executionNPVs = zeros(Float64, numExecutions)
        executionF1Scores = zeros(Float64, numExecutions)
        
        # 9. Initialize 3D array for confusion matrices from each execution
        executionConfusionMatrices = zeros(Float64, numClasses, numClasses, numExecutions)
        
        # 10. INNER LOOP: Multiple executions per fold (for non-deterministic ANNs)
        for execution in 1:numExecutions
            
            # 11. Handle validation split if validationRatio > 0
            if validationRatio > 0
                # Split training set into training and validation
                # Apply ratio directly to training subset (standard practice)
                trainIndices, valIndices = holdOut(size(trainingInputs, 1), validationRatio)
                
                execTrainingInputs = trainingInputs[trainIndices, :]
                execTrainingTargets = trainingTargets[trainIndices, :]
                execValidationInputs = trainingInputs[valIndices, :]
                execValidationTargets = trainingTargets[valIndices, :]
            else
                # No validation set
                execTrainingInputs = trainingInputs
                execTrainingTargets = trainingTargets
                execValidationInputs = trainingInputs[1:0, :]  # Empty
                execValidationTargets = trainingTargets[1:0, :]  # Empty
            end
            
            # 12. Train the ANN
            ann = trainClassANN(topology,
                (execTrainingInputs, execTrainingTargets);
                validationDataset=(execValidationInputs, execValidationTargets),
                testDataset=(testInputs, testTargets),
                transferFunctions=transferFunctions,
                maxEpochs=maxEpochs,
                minLoss=minLoss,
                learningRate=learningRate,
                maxEpochsVal=maxEpochsVal)
            
            # 13. Make predictions on test set
            testOutputs = ann(testInputs')'
            
            # 14. Calculate confusion matrix and metrics
            acc, errorRate, sens, spec, ppv, npv, f1, confMatrix = confusionMatrix(
                testOutputs, testTargets)
            
            # 15. Store metrics for this execution
            executionAccuracies[execution] = acc
            executionErrorRates[execution] = errorRate
            executionSensitivities[execution] = sens
            executionSpecificities[execution] = spec
            executionPPVs[execution] = ppv
            executionNPVs[execution] = npv
            executionF1Scores[execution] = f1
            
            # 16. Store confusion matrix for this execution
            executionConfusionMatrices[:, :, execution] = confMatrix
        end
        
        # 17. Average metrics across all executions for this fold
        testAccuracies[numFold] = mean(executionAccuracies)
        testErrorRates[numFold] = mean(executionErrorRates)
        testSensitivities[numFold] = mean(executionSensitivities)
        testSpecificities[numFold] = mean(executionSpecificities)
        testPPVs[numFold] = mean(executionPPVs)
        testNPVs[numFold] = mean(executionNPVs)
        testF1Scores[numFold] = mean(executionF1Scores)
        
        # 18. Average confusion matrices across executions
        # mean() with dims=3 returns 3D array, extract 2D slice
        foldConfusionMatrix = mean(executionConfusionMatrices, dims=3)[:, :, 1]
        
        # 19. Add to global confusion matrix
        globalConfusionMatrix .+= foldConfusionMatrix
    end
    
    # 20. Calculate mean and std across all folds for each metric
    return (mean(testAccuracies), std(testAccuracies)),
           (mean(testErrorRates), std(testErrorRates)),
           (mean(testSensitivities), std(testSensitivities)),
           (mean(testSpecificities), std(testSpecificities)),
           (mean(testPPVs), std(testPPVs)),
           (mean(testNPVs), std(testNPVs)),
           (mean(testF1Scores), std(testF1Scores)),
           globalConfusionMatrix
end


ANNCrossValidation (generic function with 1 method)